In [1]:
import pandas as pd
import numpy as np 
import sqlite3  
from datetime import datetime

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float.format', lambda x: f"{x:,.2f}")

DB_PATH = r"../customer_data/vault_chain_flow.db"
TABLE = "Customer_Transactions"

In [2]:
con = sqlite3.connect(DB_PATH)
cursor = con.cursor()

cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in the database:", cursor.fetchall())

Tables in the database: [('Customer_Transactions',)]


In [5]:
df = pd.read_sql_query(f"SELECT * FROM {TABLE}", con)
print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns.")

Loaded 580,962 rows and 14 columns.


In [6]:
df.isna().sum()

TransactionID              0
CustomerID                 0
TransactionDate            0
TransactionTime            0
TransactionType            0
TransactionAmount          0
TransactionCurrency    11846
AssetType              14475
CustomerDOB                0
CustGender                 0
CustLocation           17196
CustAccountBalance         0
WalletActivity             0
TransactionStatus          0
dtype: int64

In [9]:
# Rule 1 - AssestType drives currency 
asset_to_currency = {
    "BTC": "BTC",
    "ETH": "ETH",
    "USDT": "USDT",
    "Stablecoin": "USDT",
}

mask = df["TransactionCurrency"].isna() & df["AssetType"].isin(asset_to_currency)
df.loc[mask, "TransactionCurrency"] = df.loc[mask, "AssetType"].map(asset_to_currency)
print(f"Rule 1 filled: {mask.sum():,}")

# Rule 2 -  Crypto transaction type - currency from assest (if asset present)
mask = (df["TransactionCurrency"].isna()
        & df["TransactionType"].isin(["Crypto Buy", "Crypto Sell"])
        & df["AssetType"].notna())  
df.loc[mask, "TransactionCurrency"] = df.loc[mask, "AssetType"].map(asset_to_currency)
print(f"Rule 2 filled: {mask.sum():,}") 

# Rule 3 - Wallet - only movements default to GBP
mask = df["TransactionCurrency"].isna() & (df["WalletActivity"] == 1)
df.loc[mask, "TransactionCurrency"] = "GBP"
print(f"Rule 3 filled: {mask.sum():,}") 

# Rule 4 - Cross Border defaults to USD
mask = (df["TransactionCurrency"].isna() & (df["TransactionType"] == "Cross-Border Payment"))   
df.loc[mask, "TransactionCurrency"] = "USD"     
print(f"Rule 4 filled: {mask.sum():,}") 

# Rule 5 - Everything else defaults to GBP
mask = df["TransactionCurrency"].isna()
df.loc[mask, "TransactionCurrency"] = "GBP"
print(f"Rule 5 filled: {mask.sum():,}")

print(f"\nRemaining TransactionCurrency nulls: {df['TransactionCurrency'].isna().sum()}")

Rule 1 filled: 3,217
Rule 2 filled: 0
Rule 3 filled: 1,835
Rule 4 filled: 559
Rule 5 filled: 6,235

Remaining TransactionCurrency nulls: 0


In [10]:
currency_to_asset ={
    "GBP": "Traditional",
    "EUR": "Traditional",
    "USD": "Traditional",
    "USDT": "Stablecoin",
    "BTC": "BTC",
    "ETH": "ETH"
}

mask = df["AssetType"].isna()
df.loc[mask, "AssetType"] = df.loc[mask, "TransactionCurrency"].map(currency_to_asset)

print(f"AssetType filled from currency: {mask.sum():,}")
print("Remaining AssetType nulls:", df['AssetType'].isna().sum())

# Safety net - anything still null (shouldn't be) - Traditional
df["AssetType"]= df["AssetType"].fillna("Traditional")  

AssetType filled from currency: 14,475
Remaining AssetType nulls: 0


In [11]:
mode_per_customer = df.groupby("CustomerID")["CustLocation"].transform(
    lambda s: s.dropna().mode().iloc[0] if  s.notna().any() else np.nan
)

df["CustLocation"] = df["CustLocation"].fillna(mode_per_customer).fillna("Unknown")

print("Remaining CustLocation nulls:", df['CustLocation'].isna().sum())

Remaining CustLocation nulls: 0


In [12]:
df['CustLocation'].value_counts()

CustLocation
London                  122409
Manchester               46714
Birmingham               43160
Leeds                    35836
Glasgow                  30496
Bristol                  29452
Liverpool                28112
Edinburgh                25299
Sheffield                21724
Belfast                  20468
Newcastle                18798
Leicester                18361
Cardiff                  17852
Brighton                 17841
Southampton              17704
Nottingham               16845
International - EU       12046
Milton Keynes            11890
Coventry                 11178
Reading                  10281
Aberdeen                  9601
International - US        7955
International - APAC      6936
Unknown                      4
Name: count, dtype: int64

In [13]:
df.isna().sum()

TransactionID          0
CustomerID             0
TransactionDate        0
TransactionTime        0
TransactionType        0
TransactionAmount      0
TransactionCurrency    0
AssetType              0
CustomerDOB            0
CustGender             0
CustLocation           0
CustAccountBalance     0
WalletActivity         0
TransactionStatus      0
dtype: int64

In [14]:
df.head(10)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,London,"5,071.08",1,Completed
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed
5,TXN00000006,CUST00001,2024-12-15,07:10:38,Crypto Buy,74.81,USDT,USDT,2007-04-30,Male,London,"4,172.19",1,Completed
6,TXN00000007,CUST00001,2024-12-15,11:41:13,Cross-Border Payment,441.29,EUR,Traditional,2007-04-30,Male,London,"3,797.09",0,Completed
7,TXN00000008,CUST00001,2024-12-15,19:27:06,Deposit,275.91,EUR,Traditional,2007-04-30,Male,London,"4,031.62",0,Completed
8,TXN00000009,CUST00001,2024-12-16,17:44:56,Deposit,169.85,USD,Traditional,2007-04-30,Male,London,"4,164.10",0,Completed
9,TXN00000010,CUST00001,2024-12-21,08:17:59,P2P Transfer,182.63,USD,Traditional,2007-04-30,Male,London,"4,021.65",0,Completed


In [15]:
df.duplicated().sum()
df['TransactionID'].duplicated().sum()

np.int64(0)

In [16]:
import os
os.makedirs("../customer_data/artifacts", exist_ok=True)

In [17]:
df.to_csv("../customer_data/artifacts/cleaned_transaction_data.csv", index=False)